In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

warnings.filterwarnings('ignore')

COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

# Features seleccionadas por IG > 0.40 (Table 4 del paper)
# Features 5, 3, 6, 4, 30, 29, 33, 34 en numeración del paper
SELECTED_FEATURES = [
    'src_bytes',              # feature 5
    'service',                # feature 3
    'dst_bytes',              # feature 6
    'flag',                   # feature 4
    'diff_srv_rate',          # feature 30
    'same_srv_rate',          # feature 29
    'dst_host_srv_count',     # feature 33
    'dst_host_same_srv_rate', # feature 34
]

def find_file(filename):
    search_paths = [
        (Path.home() / ".cache" / "kagglehub" / "datasets" / "hassan06" / "nslkdd" / "versions" / "1"),
        Path("."),
        Path.home() / "Downloads",
    ]
    for base in search_paths:
        candidate = base / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontró '{filename}'.")

# Paper Section 5.2: "the detectors were trained using the NSL KDDTrain+20%
# made up of 25192 instances, 13449 normal and 11743 attack data"
df = pd.read_csv(
    find_file('KDDTrain+_20Percent.txt'),
    header=None,
    names=COL_NAMES
).drop(columns=['difficulty'])

print(f"Instancias cargadas: {len(df)}")
print(df['label'].value_counts())


Instancias cargadas: 25192
label
normal             13449
neptune             8282
ipsweep              710
satan                691
portsweep            587
smurf                529
nmap                 301
back                 196
teardrop             188
warezclient          181
pod                   38
guess_passwd          10
warezmaster            7
buffer_overflow        6
imap                   5
rootkit                4
multihop               2
phf                    2
ftp_write              1
land                   1
loadmodule             1
spy                    1
Name: count, dtype: int64


In [2]:
# Encoding de variables nominales (Section 5.2.A)
# El paper especifica: "specific values were assigned to each variable"
# TCP=1, UDP=2, ICMP=3 — único mapeo explícito del paper
protocol_map = {'tcp': 1, 'udp': 2, 'icmp': 3}
df['protocol_type'] = df['protocol_type'].str.lower().map(protocol_map).fillna(0)

# service y flag: LabelEncoder sobre todo el df
# GAP: el paper no especifica el mapeo exacto de service y flag
le = LabelEncoder()
df['service'] = le.fit_transform(df['service'].astype(str))
le = LabelEncoder()
df['flag']    = le.fit_transform(df['flag'].astype(str))

# Mapeo de etiquetas a 5 clases numéricas (Section 5.2.A)
# 1=Normal, 2=DoS, 3=Probe, 4=R2L, 5=U2R
DOS   = {'back','land','neptune','pod','smurf','teardrop',
         'apache2','udpstorm','processtable','mailbomb'}
PROBE = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L   = {'ftp_write','guess_passwd','imap','multihop','phf','spy',
         'warezclient','warezmaster','sendmail','named',
         'snmpgetattack','snmpguess','xlock','xsnoop','worm'}
U2R   = {'buffer_overflow','loadmodule','perl','rootkit',
         'httptunnel','ps','sqlattack','xterm'}

def map_label(label):
    label = label.lower().strip()
    if label == 'normal': return 1
    if label in DOS:      return 2
    if label in PROBE:    return 3
    if label in R2L:      return 4
    if label in U2R:      return 5
    return 2  # ataque no catalogado → DoS por defecto

df['label'] = df['label'].apply(map_label)

# Split 80/20 interno (Section 5, paso 2)
# "Apportioning the dataset into 20% test and 80% train"
# GAP: el paper no especifica semilla aleatoria → usamos 42
X = df[SELECTED_FEATURES].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Normalización Min-Max (Section 5.2.B)
# "min-max normalization was used to normalize the features,
#  mapping all values to the [0,1] range"
# CORREGIDO: scaler ajustado solo sobre train (post-split), sin data leakage
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

CLASSES     = sorted(np.unique(y_train))  # [1, 2, 3, 4, 5]
CLASS_NAMES = {1: 'Normal', 2: 'DoS', 3: 'Probe', 4: 'R2L', 5: 'U2R'}

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Features seleccionadas: {SELECTED_FEATURES}")


Train: 20,153 | Test: 5,039
Features seleccionadas: ['src_bytes', 'service', 'dst_bytes', 'flag', 'diff_srv_rate', 'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate']


In [3]:
from sklearn.svm import SVC

# KDENaiveBayes — replica el NaiveBayes de WEKA
class KDENaiveBayes(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, bandwidth=0.5):
        self.bandwidth = bandwidth

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.classes_ = np.unique(y)
        self.kdes_, self.priors_ = {}, {}
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / len(X)
            self.kdes_[c] = KernelDensity(bandwidth=self.bandwidth).fit(X_c)
        return self

    def predict_proba(self, X):
        X = np.array(X)
        log_p = np.array([
            np.log(self.priors_[c]) + self.kdes_[c].score_samples(X)
            for c in self.classes_
        ]).T
        log_p -= log_p.max(axis=1, keepdims=True)
        p = np.exp(log_p)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

# A1–A8: definición de clasificadores base
a1_j48 = DecisionTreeClassifier(
    criterion='entropy', min_samples_leaf=2, random_state=1
)
a2_meta_pagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(min_samples_leaf=2, random_state=1),
    n_estimators=10, max_samples=1.0, random_state=1
)
a3_random_tree = DecisionTreeClassifier(
    max_features=6, min_samples_leaf=1, random_state=1
)
a4_reptree = DecisionTreeClassifier(
    min_samples_leaf=2, random_state=1
)
a5_adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=1),
    n_estimators=10, random_state=1
)
a6_stump = DecisionTreeClassifier(max_depth=1, random_state=1)
a7_naive_bayes = KDENaiveBayes(bandwidth=0.5)
# GAP: el paper no especifica kernel ni parámetros de SVM
a8_svm = SVC(kernel='rbf', probability=True, C=1.0, gamma='scale', random_state=1)

classifiers = {
    'A1 - J48':           a1_j48,
    'A2 - Meta Pagging':  a2_meta_pagging,
    'A3 - RandomTree':    a3_random_tree,
    'A4 - REPTree':       a4_reptree,
    'A5 - AdaBoostM1':    a5_adaboost,
    'A6 - DecisionStump': a6_stump,
    'A7 - NaiveBayes':    a7_naive_bayes,
    'A8 - SVM':           a8_svm,
}

trained_clfs = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    trained_clfs[name] = clf

print("Clasificadores A1–A8 entrenados.")


Clasificadores A1–A8 entrenados.


In [4]:
import pandas as pd

def compute_weka_metrics(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    tp_rates, fp_rates, supports = [], [], []
    for i in range(len(classes)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FP - FN
        support = cm[i, :].sum()
        tp_rates.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        fp_rates.append(FP / (FP + TN) if (FP + TN) > 0 else 0.0)
        supports.append(support)
    total = sum(supports)
    tp_w = sum(tp * s for tp, s in zip(tp_rates, supports)) / total
    fp_w = sum(fp * s for fp, s in zip(fp_rates, supports)) / total
    return tp_w, fp_w

# TABLE 5
print("=" * 68)
print("TABLE 5 — TP Rate, FP Rate y Accuracy")
print("=" * 68)
print(f"{'Clasificador':<26} {'TP Rate':>9} {'FP Rate':>9} {'Accuracy':>10}")
print("-" * 68)

all_probas = []
for name, clf in trained_clfs.items():
    y_pred = clf.predict(X_test)
    acc    = accuracy_score(y_test, y_pred) * 100
    tp, fp = compute_weka_metrics(y_test, y_pred, CLASSES)
    print(f"{name:<26} {tp:>9.3f} {fp:>9.3f} {acc:>9.2f}%")

    if hasattr(clf, 'predict_proba'):
        p = clf.predict_proba(X_test)
        clf_classes = list(clf.classes_)
        p_full = np.zeros((len(X_test), len(CLASSES)))
        for j, c in enumerate(CLASSES):
            if c in clf_classes:
                p_full[:, j] = p[:, clf_classes.index(c)]
        all_probas.append(p_full)

print("-" * 68)

avg_proba  = np.mean(all_probas, axis=0)
y_pred_ens = np.array(CLASSES)[np.argmax(avg_proba, axis=1)]
acc_e      = accuracy_score(y_test, y_pred_ens) * 100
tp_e, fp_e = compute_weka_metrics(y_test, y_pred_ens, CLASSES)
print(f"{'E  - Proposed Model':<26} {tp_e:>9.3f} {fp_e:>9.3f} {acc_e:>9.2f}%")
print("=" * 68)

# TABLE 6
CLASS_T6    = [1, 2, 3, 5, 4]  # Normal, DoS, Probe, U2R, R2L
CLASS_LABEL = {1: 'Normal', 2: 'DoS', 3: 'Probe', 4: 'R2L', 5: 'U2R'}

def per_class_acc(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    return {
        c: cm[i, i] / cm[i, :].sum() * 100 if cm[i, :].sum() > 0 else 0.0
        for i, c in enumerate(classes)
    }

table6_entries = [
    ('J48',                   trained_clfs['A1 - J48']),
    ('SVM',                   trained_clfs['A8 - SVM']),
    ('Naïve Bayes',      trained_clfs['A7 - NaiveBayes']),
    ('Proposed Hybrid Model', None),
]

print("\n" + "=" * 58)
print("TABLE 6 — Accuracy por clase")
print("=" * 58)
print(f"{'Classification Algorithm':<26} {'Class Name':<12} {'Test Accuracy':>12}")
print("-" * 58)

for clf_name, clf in table6_entries:
    y_pred = clf.predict(X_test) if clf is not None else y_pred_ens
    accs = per_class_acc(y_test, y_pred, CLASSES)
    for j, c in enumerate(CLASS_T6):
        label = clf_name if j == 0 else ''
        print(f"{label:<26} {CLASS_LABEL[c]:<12} {accs[c]:>11.1f}")
    print("-" * 58)

print("=" * 58)


TABLE 5 — TP Rate, FP Rate y Accuracy
Clasificador                 TP Rate   FP Rate   Accuracy
--------------------------------------------------------------------
A1 - J48                       0.993     0.004     99.27%
A2 - Meta Pagging              0.993     0.004     99.33%
A3 - RandomTree                0.994     0.002     99.44%
A4 - REPTree                   0.993     0.005     99.27%
A5 - AdaBoostM1                0.818     0.104     81.84%
A6 - DecisionStump             0.837     0.116     83.67%


A7 - NaiveBayes                0.840     0.142     84.02%


A8 - SVM                       0.921     0.073     92.08%


--------------------------------------------------------------------
E  - Proposed Model            0.994     0.003     99.44%

TABLE 6 — Accuracy por clase
Classification Algorithm   Class Name   Test Accuracy
----------------------------------------------------------
J48                        Normal              99.6
                           DoS                 99.6
                           Probe               97.4
                           U2R                 50.0
                           R2L                 88.1
----------------------------------------------------------


SVM                        Normal              97.9
                           DoS                 88.5
                           Probe               80.8
                           U2R                  0.0
                           R2L                  7.1
----------------------------------------------------------


Naïve Bayes                Normal              96.1
                           DoS                 89.2
                           Probe                0.0
                           U2R                  0.0
                           R2L                  0.0
----------------------------------------------------------
Proposed Hybrid Model      Normal              99.7
                           DoS                 99.8
                           Probe               98.7
                           U2R                  0.0
                           R2L                 83.3
----------------------------------------------------------
